In [1]:
#Very simple NN. Raw audio -> w2v -> FC layer ->  multi-label classification


In [1]:
import torch
print(torch.version.cuda)
print(torch.cuda.is_available())

13.0
True


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('SEP-28k-Filtered_with_Split_and_Path.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28177 entries, 0 to 28176
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Unnamed: 0        28177 non-null  int64
 1   Show              28177 non-null  str  
 2   EpId              28177 non-null  int64
 3   ClipId            28177 non-null  int64
 4   SEP28k-T          28177 non-null  str  
 5   SEP28k-D          28177 non-null  str  
 6   Prolongation      28177 non-null  int64
 7   Block             28177 non-null  int64
 8   SoundRep          28177 non-null  int64
 9   WordRep           28177 non-null  int64
 10  Interjection      28177 non-null  int64
 11  NoStutteredWords  28177 non-null  int64
 12  filepath          28177 non-null  str  
dtypes: int64(9), str(4)
memory usage: 2.8 MB


In [3]:
labels = ["Prolongation","Block", "SoundRep", "WordRep", "Interjection"]

df[labels] = df[labels].astype(np.float32) #wav2vec expects floats
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28177 entries, 0 to 28176
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        28177 non-null  int64  
 1   Show              28177 non-null  str    
 2   EpId              28177 non-null  int64  
 3   ClipId            28177 non-null  int64  
 4   SEP28k-T          28177 non-null  str    
 5   SEP28k-D          28177 non-null  str    
 6   Prolongation      28177 non-null  float32
 7   Block             28177 non-null  float32
 8   SoundRep          28177 non-null  float32
 9   WordRep           28177 non-null  float32
 10  Interjection      28177 non-null  float32
 11  NoStutteredWords  28177 non-null  int64  
 12  filepath          28177 non-null  str    
dtypes: float32(5), int64(4), str(4)
memory usage: 2.3 MB


In [4]:
#Split entries based on SEP28k-T SEP
#copy so that they're their own objects
T_train_df = df[df["SEP28k-T"]== 'train'].copy()
T_dev_df = df[df["SEP28k-T"]== 'dev'].copy()
T_test_df = df[df["SEP28k-T"]== 'test'].copy()

In [23]:
#make sample smaller just to see if training works, change these values as necessary
T_train_df = T_train_df.sample(n=min(500, len(T_train_df)), random_state=42)
T_dev_df   = T_dev_df.sample(n=min(100,  len(T_dev_df)),   random_state=42)
T_test_df  = T_test_df.sample(n=min(100,  len(T_test_df)), random_state=42)

In [6]:
print(len(T_train_df), len(T_dev_df), len(T_test_df))

500 100 100


In [7]:
# pytorch setup
import torch
import torch.nn as nn
import torchaudio
import soundfile as sf
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
labels = ["Prolongation","Block", "SoundRep", "WordRep", "Interjection"]
sample_rate = 16000
batch_size = 8
learning_rate = 1e-3
target_sr = 16000 
epochs = 10

In [8]:
w2v_starter = torchaudio.pipelines.WAV2VEC2_BASE
w2v_model = w2v_starter.get_model().to(device)

In [9]:
def load_wav(path): #replacing torchload with soundfile
    x, sr = sf.read(path, dtype="float32", always_2d=True)  # (T, C)
    x = torch.from_numpy(x).transpose(0, 1)                 # (C, T)
    return x, sr

In [10]:
#Create a dataset for audio files and labels (based on Mohapatra et al.)
class SEP_DataSet (Dataset):
    def __init__(self, dframe,labels, target_sr):
        self.df = dframe.reset_index(drop=True) #because the indices after we separate into test/dev/train is no longer in order
        self.labels = labels
        self.target_sr = target_sr # we want this to be 16k, since wav2vec expects it 

    def __len__(self):
        return len(self.df)

    def __getitem__(self,index) : #for each raw wav, will return a raw tensor representation 
        row = self.df.iloc[index] #take one row from df
        path = row["filepath"]  #filepath of wav file
        wav, sr = load_wav(path) #wav is raw tensor (should be (1,48000)), sr is sampling rate
        if wav.size(0)> 1:
            wav = wav.mean(dim=0, keepdim=True) #convert to mono audio if there are more than 1 channel
        if sr != self.target_sr: #if raw audio is not 16k hz, resample to 16k.
            wav = torchaudio.functional.resample(wav, sr, self.target_sr)
        wav = wav.squeeze(0) #remove first dimension after ensuring it's mono. 
        #label_tensor = torch.tensor(row[self.labels].values, dtype=torch.float32) #converting label colums into one tensor. 
        vals = row[self.labels]  # pandas Series

        # force numeric; anything weird becomes NaN
        vals = pd.to_numeric(vals, errors="coerce").to_numpy(dtype=np.float32)
        vals = np.nan_to_num(vals, nan=0.0)
        label_tensor = torch.from_numpy(vals)
        return wav, label_tensor

In [11]:
#data loaders
train_loader = DataLoader(SEP_DataSet(T_train_df, labels, target_sr),
                          batch_size=batch_size, shuffle=True)

valid_loader = DataLoader(SEP_DataSet(T_dev_df, labels, target_sr),
                          batch_size=batch_size, shuffle=True)

test_loader = DataLoader(SEP_DataSet(T_test_df, labels, target_sr),
                         batch_size=batch_size, shuffle=False)

In [12]:
#Simple wav2vec --> FCC layer
w2v_model = w2v_starter.get_model().to(device)

for p in w2v_model.parameters(): #freezing the parameters so that it doesn't get optimized. Essensitally the w2v part is just a preprocessing step.
    p.requires_grad = False
    
class FC_Module (nn.Module):
    def __init__(self, w2v_model, num_labels=5, hidden_dim = 256, dropout=0.1):
        super().__init__()
        self.w2v = w2v_model
        in_dim = 768  # input dim (which is the 768 output from w2v)
        
        self.classifier = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_labels)
        )
        
    def forward(self, wavs):
        if wavs.dim() == 3:
            avs = wavs.squeeze(1)
        features, _ = self.w2v(wavs)   # torchaudio returns a tuple
        pooled = features.mean(dim=1)  # average over time
        out = self.classifier(pooled)
        return out
model = FC_Module(w2v_model, num_labels=5).to(device)

In [13]:
criterion = nn.BCEWithLogitsLoss() #Cross entropy for multi-label classification.

optimizer = torch.optim.AdamW(
    model.classifier.parameters(),
    lr=learning_rate,
)


In [14]:
#Training loop

for epoch in range(epochs):
    model.train()
    train_loss = 0.0 #to keep track of for each batch
    
    for wavs, y in train_loader: 
        wavs = wavs.to(device)
        y = y.to(device)
        optimizer.zero_grad()

        out = model(wavs)
        loss = criterion(out,y)
    
        loss.backward()
        optimizer.step()
    
        train_loss += loss.item() * wavs.size(0) #adding to total loss

    train_loss /= len(train_loader.dataset) #mean loss within each epoch

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for wavs, y in valid_loader:
            wavs = wavs.to(device)
            y = y.to(device)

            out = model(wavs)
            loss = criterion(out, y)

            val_loss += loss.item() * wavs.size(0)

    val_loss /= len(valid_loader.dataset)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}"
    )

Epoch [1/10] Train Loss: 0.6543 | Val Loss: 0.6622
Epoch [2/10] Train Loss: 0.6269 | Val Loss: 0.6885
Epoch [3/10] Train Loss: 0.6045 | Val Loss: 0.6684
Epoch [4/10] Train Loss: 0.5865 | Val Loss: 0.6494
Epoch [5/10] Train Loss: 0.5726 | Val Loss: 0.6584
Epoch [6/10] Train Loss: 0.5511 | Val Loss: 0.6369
Epoch [7/10] Train Loss: 0.5130 | Val Loss: 0.7090
Epoch [8/10] Train Loss: 0.4912 | Val Loss: 0.6722
Epoch [9/10] Train Loss: 0.4748 | Val Loss: 0.5938
Epoch [10/10] Train Loss: 0.4551 | Val Loss: 0.6433


In [15]:
#running test set
labels = ["Prolongation", "Block", "SoundRep", "WordRep", "Interjection"]

model.eval()
all_probs = []
with torch.no_grad():
    for x_batch, _ in test_loader:
        x_batch = x_batch.to(device)
        logits = model(x_batch)                 
        probs = torch.sigmoid(logits)           
        all_probs.append(probs.cpu())


all_probs = torch.cat(all_probs, dim=0)
probs_np = all_probs.numpy()

In [16]:
#saving model
MODEL_PATH = "embedded_nn_model.pt"

torch.save(model.state_dict(), MODEL_PATH)

print(f"Model saved to {MODEL_PATH}")

Model saved to embedded_nn_model.pt


In [17]:
T_test_df.head()

,Unnamed: 0,Show,EpId,ClipId,SEP28k-T,SEP28k-D,Prolongation,Block,SoundRep,WordRep,Interjection,NoStutteredWords,filepath
1534,1534,HeStutters,11,110,test,test,0.0,0.0,2.0,1.0,0.0,0,SEP-28k_CLIP/HeStutters/11/HeStutters_11_110.wav
23310,23310,WomenWhoStutter,50,116,test,test,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/WomenWhoStutter/50/WomenWhoStutte...
18797,18797,StutteringIsCool,82,35,test,test,1.0,2.0,0.0,0.0,3.0,2,SEP-28k_CLIP/StutteringIsCool/82/StutteringIsC...
26558,26558,WomenWhoStutter,90,10,test,test,3.0,0.0,0.0,0.0,0.0,0,SEP-28k_CLIP/WomenWhoStutter/90/WomenWhoStutte...
21620,21620,WomenWhoStutter,29,37,test,test,0.0,2.0,0.0,0.0,0.0,1,SEP-28k_CLIP/WomenWhoStutter/29/WomenWhoStutte...


In [18]:
def clipName (row):
    show = str(row["Show"])
    episode = str(row["EpId"])
    clip = str(row["ClipId"])

    return f"{show}_{episode}_{clip}"

In [19]:
probs_df = pd.DataFrame(probs_np, columns=labels)

In [20]:
clips = T_test_df.apply(clipName, axis=1)
probs_df.insert(0, "Clip", clips.values)

In [21]:
print(probs_df.head())

                     Clip  Prolongation     Block  SoundRep   WordRep  \
0       HeStutters_11_110      0.199349  0.591466  0.297537  0.423477   
1  WomenWhoStutter_50_116      0.464526  0.230076  0.135993  0.292203   
2  StutteringIsCool_82_35      0.503790  0.433540  0.161460  0.240452   
3   WomenWhoStutter_90_10      0.416386  0.735803  0.241465  0.262458   
4   WomenWhoStutter_29_37      0.527490  0.379625  0.371775  0.364449   

   Interjection  
0      0.900898  
1      0.959526  
2      0.987696  
3      0.516608  
4      0.956227  


In [22]:
probs_df.to_csv("SEP28k-simpleW2V-testprob.csv", index=False)
